https://geemap.org/notebooks/114_dynamic_world/

https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1#colab-python

https://developers.google.com/earth-engine/tutorials/community/introduction-to-dynamic-world-pt-1

# **0 Import Library**

In [1]:
!pip install rasterio

In [2]:
!pip install -U geemap geedim


In [26]:
!pip install mss

In [1]:
import ee
import geemap
import geopandas as gpd
import rasterio
import os

In [2]:
ee.Authenticate()
ee.Initialize()  #project= "valued-mediator-463106-q8")


Successfully saved authorization token.


# **1 Input the polygon shape file**

In [4]:
gis_dir = '/home/luxizhou/GIS/Admin_Boundaries/PHL_admin'

shp = gpd.read_file(os.path.join(gis_dir,"PHL_adm2.shp"))
shapefile_path = os.path.join(os.path.join(gis_dir,"Metro_Manila_Box.shp"))
gdf = gpd.read_file(shapefile_path)

In [7]:
gdf

,MINX,MINY,MAXX,MAXY,CNTX,CNTY,AREA,PERIM,HEIGHT,WIDTH,geometry
0,120.942223,14.560531,121.023697,14.6385,120.98296,14.599515,0.006353,0.318888,0.07797,0.081474,"POLYGON ((120.39379 14.07534, 120.39379 14.946..."


In [8]:
# Function to extract coordinates from a geometry
def extract_coordinates(geometry):
    coords = []
    if geometry.geom_type == 'Polygon':
        coords.extend(list(geometry.exterior.coords))
    elif geometry.geom_type == 'MultiPolygon':
        for poly in geometry.geoms:
            coords.extend(list(poly.exterior.coords))
    elif geometry.geom_type == 'Point':
        coords.append((geometry.x, geometry.y))
    return coords

# Collect all coordinates using a for loop
all_coordinates = []
print("Longitude,Latitude")
for index, row in gdf.iterrows():
    coords = extract_coordinates(row.geometry)
    for lon, lat in coords:
        print(f"{lon},{lat}")
        all_coordinates.append((lon, lat))


# Extract first and third coordinates
first_coord = all_coordinates[0]  # First coordinate
third_coord = all_coordinates[2]  # Third coordinate
print(first_coord, third_coord)

Longitude,Latitude
120.39379207352661,14.075338118147869
120.39379207352661,14.946778538280228
121.30440398410013,14.946778538280228
121.30440398410013,14.075338118147869
120.39379207352661,14.075338118147869
(120.39379207352661, 14.075338118147869) (121.30440398410013, 14.946778538280228)


In [9]:
# Create ee.Geometry.BBox (minLon, minLat, maxLon, maxLat)
region = ee.Geometry.BBox(
    first_coord[0],  # minLon (first Longitude)
    first_coord[1],  # minLat (first Latitude)
    third_coord[0],  # maxLon (third Longitude)
    third_coord[1]   # maxLat (third Latitude)
)

# **2. Input the Time Period**

In [10]:
START = ee.Date('2015-06-27')
END = ee.Date('2016-06-27')
START_2 = ee.Date('2024-07-26')
END_2 = ee.Date('2025-07-26')

# **3.1 First Time Period in all attributes of LandCover**

In [11]:
Map = geemap.Map()
Map.centerObject(region, zoom=9)  # Center the map on the region with appropriate zoom level
image = geemap.dynamic_world_s2(region, START, END)
vis_params = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
Map.addLayer(image, vis_params, "Sentinel-2 image")

landcover = geemap.dynamic_world(region, START, END, return_type="hillshade")
Map.addLayer(landcover, {}, "Land Cover")
Map.add_legend(title="Dynamic World Land Cover", builtin_legend="Dynamic_World")
Map

Map(center=[14.510918728181457, 120.84909802881334], controls=(WidgetControl(options=['position', 'transparent…

# **3.2 Second Time Period in all attributes of LandCover**

In [12]:
Map2 = geemap.Map()
Map2.centerObject(region, zoom=9)  # Center the map on the region with appropriate zoom level
image = geemap.dynamic_world_s2(region, START_2, END_2)
vis_params = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
Map2.addLayer(image, vis_params, "Sentinel-2 image")

landcover = geemap.dynamic_world(region, START_2, END_2, return_type="hillshade")
Map2.addLayer(landcover, {}, "Land Cover")
Map2.add_legend(title="Dynamic World Land Cover", builtin_legend="Dynamic_World")
Map2

Map(center=[14.510918728181457, 120.84909802881334], controls=(WidgetControl(options=['position', 'transparent…

# **3.3 find the difference of different land use type between the first time period and the second time period**

In [13]:
bands = [
    'water',
    'trees',
    'grass',
    'flooded_vegetation',
    'crops',
    'shrub_and_scrub',
    'built',
    'bare',
    'snow_and_ice'
]
colors = [
    '419bdf',
    '397d49',
    '88b053',
    '7a87c6',
    'e49635',
    'dfc35a',
    'c4281b',
    'a59b8f',
    'b39fe1',
]

In [14]:
Map3 = geemap.Map()
Map3.centerObject(region, zoom=9)  # Center the map on the region with appropriate zoom level

colFilter = ee.Filter.And(
    ee.Filter.bounds(region),
    ee.Filter.date(START, END))

colFilter_2024_2025 = ee.Filter.And(
   ee.Filter.bounds(region),
   ee.Filter.date(START_2, END_2))


dwCol = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter)
dwCol_2024_2025 = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter_2024_2025)

for band, color in zip(bands, colors):
    # Calculate max for each period
    band_2015_2016 = dwCol.select(band).reduce(ee.Reducer.max())
    band_2024_2025 = dwCol_2024_2025.select(band).reduce(ee.Reducer.max())

    # Calculate difference
    diff_band = band_2024_2025.subtract(band_2015_2016)

    diff_band = diff_band.clamp(0, 1)

    # Visualization parameters
    diff_vis = {
        'min': 0,
        'max': 1,
        'palette': ['#FFFFFF', color]
    }

    Map3.addLayer(diff_band, diff_vis, f'New {band} Areas')

Map3


Map(center=[14.510918728181457, 120.84909802881334], controls=(WidgetControl(options=['position', 'transparent…

# **4 Take the bare (bands) as an example**

Below two map can see the building of the New Manila International Airport start from 2020 on the coastal areas of Bulakan, Bulacan

In [15]:
Map4 = geemap.Map()
Map4.centerObject(region, zoom=9)  # Center the map on the region with appropriate zoom level

colFilter = ee.Filter.And(
    ee.Filter.bounds(region),
    ee.Filter.date(START, END))

dwCol = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter)
built = dwCol.select('bare').reduce(ee.Reducer.max())
diff_built_vis = {
  'min': 0,
  'max': 1,
  'palette': ['#FFFFFF', '#a59b8f']
}
Map4.addLayer(built, diff_built_vis, 'Old Areas')
legend_dict = {
   'No Change': '#FFFFFF',
   'Old Areas': '#a59b8f'
}
Map4.add_legend(legend_title="Old Areas", legend_dict=legend_dict)
Map4

Map(center=[14.510918728181457, 120.84909802881334], controls=(WidgetControl(options=['position', 'transparent…

In [16]:
Map5 = geemap.Map()
Map5.centerObject(region, zoom=9)  # Center the map on the region with appropriate zoom level

colFilter = ee.Filter.And(
    ee.Filter.bounds(region),
    ee.Filter.date(START_2, END_2))

dwCol = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter)
built = dwCol.select('bare').reduce(ee.Reducer.max())
diff_built_vis = {
  'min': 0,
  'max': 1,
  'palette': ['#FFFFFF', '#a59b8f']
}
Map5.addLayer(built, diff_built_vis, 'New Areas')
legend_dict = {
   'No Change': '#FFFFFF',
   'New Areas': '#a59b8f'
}
Map5.add_legend(legend_title="New Areas", legend_dict=legend_dict)
Map5

Map(center=[14.510918728181457, 120.84909802881334], controls=(WidgetControl(options=['position', 'transparent…